In [ ]:
# Only this cell fixes the numpy/scipy/sklearn clash. Do NOT import sklearn here — restart first.
import subprocess
import sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", *args], stdout=subprocess.DEVNULL)

pip("install", "-q", "-U", "pip")
for pkg in ("numpy", "scipy", "scikit-learn"):
    pip("uninstall", "-y", pkg)
pip("install", "-q", "--no-cache-dir", "numpy==2.2.6")
pip("install", "-q", "--no-cache-dir", "scipy==1.15.2", "scikit-learn==1.6.1")
pip("install", "-q", "fastapi", "uvicorn", "pyngrok", "nest-asyncio", "python-multipart")
pip("install", "-q", "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git")
pip("install", "-q", "--force-reinstall", "--no-deps", "numpy==2.2.6")
pip("install", "-q", "--no-cache-dir", "scipy==1.15.2", "scikit-learn==1.6.1")

print("Install finished.")
print(">>> Session → Restart session, then run cells 2 onward (skip this cell). <<<")

In [ ]:
import os

# Kaggle: Add-ons → Secrets → HF_TOKEN, NGROK_TOKEN, NGROK_DOMAIN
HF_TOKEN = os.environ.get('HF_TOKEN', '')
NGROK_TOKEN = os.environ.get('NGROK_TOKEN', '')
NGROK_DOMAIN = os.environ.get('NGROK_DOMAIN', '')  # hostname only, e.g. xxx.ngrok-free.dev

if not all((HF_TOKEN, NGROK_TOKEN, NGROK_DOMAIN)):
    raise ValueError('Set HF_TOKEN, NGROK_TOKEN, and NGROK_DOMAIN as Kaggle secrets or env vars')

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '300'

from pyngrok import conf
conf.get_default().auth_token = NGROK_TOKEN
print('Credentials set')

Credentials set


In [2]:
# Preflight: catches numpy/scipy mismatch before loading the ~7 GB model
import numpy as np
from scipy.sparse import issparse
from sklearn.model_selection import GroupShuffleSplit

from tribev2 import TribeModel

CACHE_DIR = '/kaggle/working/tribe_cache'
print('Loading TRIBE v2 — first run downloads ~7 GB, subsequent runs use cache...')
tribe_model = TribeModel.from_pretrained('facebook/tribev2', cache_folder=CACHE_DIR)
print('TRIBE v2 ready')

/usr/local/lib/python3.12/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-05-19 00:21:22 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.


Loading TRIBE v2 — first run downloads ~7 GB, subsequent runs use cache...


config.yaml: 0.00B [00:00, ?B/s]

best.ckpt:   0%|          | 0.00/709M [00:00<?, ?B/s]

2026-05-19 00:21:29 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
INFO - Loading model from /root/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt
/usr/local/lib/python3.12/dist-packages/x_transformers/x_transformers.py:439: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/usr/local/lib/python3.12/dist-packages/x_transformers/x_transformers.py:461: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


TRIBE v2 ready


In [3]:
import hashlib
import json
import os
import tempfile
from typing import Optional

import numpy as np
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
from pydantic import BaseModel

try:
    import nibabel as nib
    from nilearn import datasets
    _VIEWER_IMPORT_ERROR = None
except Exception as e:
    nib = None
    datasets = None
    _VIEWER_IMPORT_ERROR = str(e)

TRIBE_API_VERSION = '2.5'
VIEWER_DIR = '/kaggle/working/tribe_viewers'
VIEWER_MAX_FRAMES = int(os.environ.get('TRIBE_VIEWER_MAX_FRAMES', '120'))
os.makedirs(VIEWER_DIR, exist_ok=True)

app = FastAPI(title='TRIBE v2 Activation API', version=TRIBE_API_VERSION)
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

VIDEO_EXT = {'.mp4', '.mov', '.avi', '.mkv', '.webm'}
AUDIO_EXT = {'.wav', '.mp3', '.flac', '.m4a', '.ogg', '.aac'}

_text_cache = {}  # sha256(text) -> (preds, segments)
_viewer_cache = {}  # analysis_id -> HTML file path

# Approximate fsaverage5 vertex slices from the original notebook.
# These are useful product/demo signals, not atlas-grade neuroscience regions.
REGION_MASKS = {
    'visual_cortex': (1000, 2000),
    'language_network': (3000, 4500),
    'attention': (5000, 6000),
    'emotional_response': (6500, 7500),
    'memory_encoding': (8000, 9000),
}


def _segments_json(segments) -> list:
    if segments is None:
        return []
    try:
        import pandas as pd
        if isinstance(segments, pd.DataFrame):
            return segments.to_dict(orient='records')
    except ImportError:
        pass
    if isinstance(segments, list):
        out = []
        for s in segments:
            if isinstance(s, dict):
                out.append(s)
            elif hasattr(s, 'start') and hasattr(s, 'end'):
                out.append({'start': float(s.start), 'end': float(s.end)})
            else:
                out.append({'value': str(s)})
        return out
    return [{'value': str(segments)}]


def _viewer_preds(activation: np.ndarray) -> list:
    """Per-frame 0–1 normalization (|value|) for BrainViewer.jsx."""
    out = []
    for frame in np.asarray(activation, dtype=np.float64):
        v = np.abs(frame)
        mx = float(v.max())
        out.append((v / mx).tolist() if mx > 0 else v.tolist())
    return out


def _region_scores(preds: np.ndarray) -> dict:
    mean_act = np.abs(preds).mean(axis=0)
    scores = {}
    for region, (start, end) in REGION_MASKS.items():
        bounded_start = max(0, min(start, mean_act.shape[0]))
        bounded_end = max(bounded_start, min(end, mean_act.shape[0]))
        region_value = mean_act[bounded_start:bounded_end].mean() if bounded_end > bounded_start else 0.0
        scores[region] = round(float(np.clip(region_value * 500, 0, 100)), 1)
    scores['overall_impact'] = round(float(np.mean(list(scores.values()))), 1)
    return scores


def _activation_summary(preds: np.ndarray) -> dict:
    scores = _region_scores(preds)
    return {
        'scores': scores,
        'region_masks': REGION_MASKS,
        'peak_activation_step': int(np.abs(preds).mean(axis=1).argmax()),
    }


def _analysis_id(preds: np.ndarray, input_type: str, metadata: dict | None = None) -> str:
    hasher = hashlib.sha256()
    hasher.update(np.asarray(preds, dtype=np.float32).tobytes())
    hasher.update(input_type.encode('utf-8'))
    if metadata:
        hasher.update(repr(sorted(metadata.items())).encode('utf-8'))
    return hasher.hexdigest()[:20]


def _viewer_frame_indices(preds: np.ndarray, input_type: str, peak_step: int) -> tuple[np.ndarray, int]:
    n_frames = int(np.asarray(preds).shape[0])
    if n_frames <= 0:
        return np.array([], dtype=int), 1

    if input_type == 'text':
        safe_peak = int(np.clip(peak_step, 0, n_frames - 1))
        return np.array([safe_peak], dtype=int), 0

    if n_frames <= VIEWER_MAX_FRAMES:
        return np.arange(n_frames, dtype=int), 1

    indices = np.linspace(0, n_frames - 1, VIEWER_MAX_FRAMES).round().astype(int)
    indices = np.unique(indices)
    stride = max(1, int(np.ceil(n_frames / max(len(indices), 1))))
    return indices, stride


def _normalize_viewer_frames(preds: np.ndarray, frame_indices: np.ndarray) -> list[list[float]]:
    frames = np.abs(np.asarray(preds, dtype=np.float32))[frame_indices]
    if frames.size == 0:
        return []

    # Match the original demo spirit: robust scaling keeps one outlier from washing out the brain.
    hi = np.percentile(frames, 99)
    if hi <= 0:
        hi = float(frames.max()) if frames.max() > 0 else 1.0
    frames = np.clip(frames / hi, 0, 1)
    return np.round(frames, 4).tolist()


def _load_half_inflated_mesh() -> dict:
    if _VIEWER_IMPORT_ERROR:
        raise RuntimeError(f'viewer dependencies are unavailable: {_VIEWER_IMPORT_ERROR}')

    fsavg = datasets.fetch_surf_fsaverage(mesh='fsaverage5')
    hemis = {}
    for hemi in ('left', 'right'):
        infl_xyz = nib.load(getattr(fsavg, f'infl_{hemi}')).darrays[0].data.astype(np.float32)
        pial_img = nib.load(getattr(fsavg, f'pial_{hemi}'))
        pial_xyz = pial_img.darrays[0].data.astype(np.float32)
        faces = pial_img.darrays[1].data.astype(np.int32)
        sulc = nib.load(getattr(fsavg, f'sulc_{hemi}')).darrays[0].data.astype(np.float32)

        # TRIBE's PlotBrain uses a half-inflated surface for the smoother demo look.
        coords = 0.5 * infl_xyz + 0.5 * pial_xyz
        if hemi == 'left':
            coords[:, 0] = coords[:, 0] - coords[:, 0].max()
        else:
            coords[:, 0] = coords[:, 0] - coords[:, 0].min()
        hemis[hemi] = {'coords': coords, 'faces': faces, 'sulc': sulc}

    left_n = hemis['left']['coords'].shape[0]
    coords = np.concatenate([hemis['left']['coords'], hemis['right']['coords']], axis=0)
    faces = np.concatenate([hemis['left']['faces'], hemis['right']['faces'] + left_n], axis=0)
    sulc = np.concatenate([hemis['left']['sulc'], hemis['right']['sulc']], axis=0)

    center = coords.mean(axis=0, keepdims=True)
    coords = coords - center
    scale = np.abs(coords).max() or 1.0
    coords = coords / scale * 5.5

    sulc_norm = (sulc - sulc.min()) / (sulc.max() - sulc.min() + 1e-8)
    # Inverted sulcal grayscale, similar to the original PyVista background blend.
    bg = np.clip(0.18 + (1 - sulc_norm) * 0.48, 0, 1)
    bg_rgb = np.stack([bg, bg, bg], axis=1)

    return {
        'coords': np.round(coords, 5).tolist(),
        'faces': faces.astype(int).tolist(),
        'bg': np.round(bg_rgb, 4).tolist(),
    }


def _build_viewer_html(preds: np.ndarray, analysis_id: str, peak_step: int, input_type: str) -> tuple[str, dict]:
    frame_indices, stride = _viewer_frame_indices(preds, input_type, peak_step)
    if len(frame_indices) == 0:
        raise RuntimeError('Not enough frames to render the cortical surface viewer.')

    peak_viewer_frame = int(np.where(frame_indices == peak_step)[0][0]) if peak_step in set(frame_indices.tolist()) else 0
    viewer_meta = {
        'viewer_kind': 'threejs_one_piece_static' if len(frame_indices) <= 1 else 'threejs_one_piece_animated',
        'viewer_frame_count': int(len(frame_indices)),
        'viewer_source_frame_count': int(np.asarray(preds).shape[0]),
        'viewer_stride': int(stride),
        'viewer_peak_frame': peak_viewer_frame,
    }

    cached = _viewer_cache.get(analysis_id)
    if cached and os.path.exists(cached):
        return cached, {**viewer_meta, 'viewer_cache_hit': True}

    html_path = os.path.join(VIEWER_DIR, f'{analysis_id}.html')
    if os.path.exists(html_path):
        _viewer_cache[analysis_id] = html_path
        return html_path, {**viewer_meta, 'viewer_cache_hit': True}

    frames = _normalize_viewer_frames(preds, frame_indices)
    if not frames:
        raise RuntimeError('Not enough frames to render the cortical surface viewer.')

    mesh = _load_half_inflated_mesh()
    if len(frames[0]) != len(mesh['coords']):
        raise RuntimeError(f'Activation vertices ({len(frames[0])}) do not match mesh vertices ({len(mesh["coords"])}).')

    payload = {
        'analysisId': analysis_id,
        'inputType': input_type,
        'coords': mesh['coords'],
        'faces': mesh['faces'],
        'bg': mesh['bg'],
        'frames': frames,
        'sourceFrameIndices': frame_indices.astype(int).tolist(),
        'peakActivationStep': int(peak_step),
        'peakViewerFrame': peak_viewer_frame,
        'stride': int(stride),
        'isStatic': len(frames) <= 1,
    }
    payload_json = json.dumps(payload, separators=(',', ':'))

    html = f"""<!doctype html>
<html>
<head>
  <meta charset=\"utf-8\" />
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\" />
  <title>TRIBE v2 Animated Brain Viewer</title>
  <style>
    html, body {{ margin: 0; height: 100%; overflow: hidden; background: #05070c; color: #e8ecf1; font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; }}
    #app {{ position: relative; width: 100vw; height: 100vh; }}
    #viewer {{ position: absolute; inset: 0; }}
    .panel {{ position: absolute; left: 16px; right: 16px; bottom: 16px; display: flex; align-items: center; gap: 12px; padding: 12px 14px; border: 1px solid rgba(148, 163, 184, 0.24); border-radius: 16px; background: rgba(5, 7, 12, 0.76); backdrop-filter: blur(12px); box-shadow: 0 20px 60px rgba(0, 0, 0, 0.35); }}
    .title {{ position: absolute; left: 16px; top: 16px; padding: 10px 12px; border: 1px solid rgba(148, 163, 184, 0.18); border-radius: 14px; background: rgba(5, 7, 12, 0.66); backdrop-filter: blur(10px); }}
    .eyebrow {{ margin: 0 0 2px; color: #94a3b8; font-size: 10px; letter-spacing: .16em; text-transform: uppercase; }}
    .name {{ margin: 0; font-size: 14px; font-weight: 650; }}
    button {{ border: 1px solid rgba(148, 163, 184, .28); border-radius: 999px; background: rgba(15, 23, 42, .9); color: #f8fafc; cursor: pointer; font: inherit; padding: 8px 14px; }}
    button:disabled {{ cursor: default; color: #64748b; }}
    input[type=range] {{ flex: 1; accent-color: #f97316; }}
    .readout {{ min-width: 172px; color: #cbd5e1; font-size: 12px; text-align: right; font-variant-numeric: tabular-nums; }}
    .hint {{ position: absolute; right: 16px; top: 16px; max-width: 280px; color: #94a3b8; font-size: 12px; text-align: right; }}
  </style>
</head>
<body>
  <div id=\"app\">
    <div id=\"viewer\"></div>
    <div class=\"title\">
      <p class=\"eyebrow\">TRIBE v2</p>
      <p class=\"name\">One-piece cortical activation</p>
    </div>
    <div class=\"hint\">Drag to rotate · Scroll to zoom · Colors animate predicted activation</div>
    <div class=\"panel\">
      <button id=\"play\">Play</button>
      <input id=\"slider\" type=\"range\" min=\"0\" max=\"0\" value=\"0\" />
      <div id=\"readout\" class=\"readout\"></div>
    </div>
  </div>
  <script id=\"payload\" type=\"application/json\">{payload_json}</script>
  <script src=\"https://cdn.jsdelivr.net/npm/three@0.128.0/build/three.min.js\"></script>
  <script src=\"https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/controls/OrbitControls.js\"></script>
  <script>
    const data = JSON.parse(document.getElementById('payload').textContent);
    const container = document.getElementById('viewer');
    const playButton = document.getElementById('play');
    const slider = document.getElementById('slider');
    const readout = document.getElementById('readout');

    const scene = new THREE.Scene();
    scene.background = new THREE.Color(0x05070c);

    const camera = new THREE.PerspectiveCamera(38, window.innerWidth / window.innerHeight, 0.01, 1000);
    camera.position.set(0, -14, 6.5);

    const renderer = new THREE.WebGLRenderer({{ antialias: true }});
    renderer.setPixelRatio(Math.min(window.devicePixelRatio || 1, 2));
    renderer.setSize(window.innerWidth, window.innerHeight);
    container.appendChild(renderer.domElement);

    const controls = new THREE.OrbitControls(camera, renderer.domElement);
    controls.enableDamping = true;
    controls.dampingFactor = 0.08;
    controls.target.set(0, 0, 0);

    scene.add(new THREE.HemisphereLight(0xffffff, 0x1f2937, 1.4));
    const keyLight = new THREE.DirectionalLight(0xffffff, 1.1);
    keyLight.position.set(4, -6, 8);
    scene.add(keyLight);
    const rimLight = new THREE.DirectionalLight(0xff9f43, 0.45);
    rimLight.position.set(-6, 4, 5);
    scene.add(rimLight);

    const positions = new Float32Array(data.coords.flat());
    const indices = new Uint32Array(data.faces.flat());
    const colors = new Float32Array(data.coords.length * 3);

    const geometry = new THREE.BufferGeometry();
    geometry.setAttribute('position', new THREE.BufferAttribute(positions, 3));
    geometry.setAttribute('color', new THREE.BufferAttribute(colors, 3));
    geometry.setIndex(new THREE.BufferAttribute(indices, 1));
    geometry.computeVertexNormals();

    const material = new THREE.MeshStandardMaterial({{
      vertexColors: true,
      roughness: 0.48,
      metalness: 0.03,
      side: THREE.DoubleSide,
    }});
    const brain = new THREE.Mesh(geometry, material);
    brain.rotation.x = -0.18;
    brain.rotation.z = 0.02;
    scene.add(brain);

    const frameCount = data.frames.length;
    let frame = Math.min(data.peakViewerFrame || 0, frameCount - 1);
    let playing = false;
    let lastAdvance = performance.now();
    const frameIntervalMs = 900;

    function clamp01(v) {{ return Math.max(0, Math.min(1, v)); }}
    function smoothstep(edge0, edge1, x) {{
      const t = clamp01((x - edge0) / (edge1 - edge0));
      return t * t * (3 - 2 * t);
    }}
    function fireColor(v) {{
      v = clamp01(v);
      if (v < 0.25) {{
        const t = v / 0.25;
        return [0.12 + 0.55 * t, 0.03 * t, 0.02];
      }}
      if (v < 0.58) {{
        const t = (v - 0.25) / 0.33;
        return [0.67 + 0.33 * t, 0.04 + 0.36 * t, 0.02];
      }}
      const t = (v - 0.58) / 0.42;
      return [1.0, 0.40 + 0.55 * t, 0.04 + 0.70 * t];
    }}
    function setFrame(nextFrame) {{
      frame = Math.max(0, Math.min(frameCount - 1, nextFrame));
      const values = data.frames[frame];
      for (let i = 0; i < values.length; i++) {{
        const v = values[i];
        const bg = data.bg[i];
        const hot = fireColor(v);
        const alpha = smoothstep(0.08, 0.42, v);
        colors[i * 3] = bg[0] * (1 - alpha) + hot[0] * alpha;
        colors[i * 3 + 1] = bg[1] * (1 - alpha) + hot[1] * alpha;
        colors[i * 3 + 2] = bg[2] * (1 - alpha) + hot[2] * alpha;
      }}
      geometry.attributes.color.needsUpdate = true;
      slider.value = String(frame);
      const sourceFrame = data.sourceFrameIndices[frame] ?? frame;
      const peak = sourceFrame === data.peakActivationStep ? ' · peak' : '';
      readout.textContent = data.isStatic
        ? `frame ${{sourceFrame}} · static peak view`
        : `frame ${{frame + 1}}/${{frameCount}} · source t=${{sourceFrame}}${{peak}}`;
    }}

    slider.max = String(Math.max(frameCount - 1, 0));
    slider.disabled = data.isStatic;
    playButton.disabled = data.isStatic;
    playButton.textContent = data.isStatic ? 'Static' : 'Play';
    slider.addEventListener('input', () => {{
      playing = false;
      playButton.textContent = 'Play';
      setFrame(Number(slider.value));
    }});
    playButton.addEventListener('click', () => {{
      if (data.isStatic) return;
      playing = !playing;
      playButton.textContent = playing ? 'Pause' : 'Play';
      lastAdvance = performance.now();
    }});

    function animate(now) {{
      requestAnimationFrame(animate);
      if (playing && now - lastAdvance > frameIntervalMs) {{
        setFrame((frame + 1) % frameCount);
        lastAdvance = now;
      }}
      controls.update();
      renderer.render(scene, camera);
    }}
    window.addEventListener('resize', () => {{
      camera.aspect = window.innerWidth / window.innerHeight;
      camera.updateProjectionMatrix();
      renderer.setSize(window.innerWidth, window.innerHeight);
    }});

    setFrame(frame);
    requestAnimationFrame(animate);
  </script>
</body>
</html>
"""

    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html)

    _viewer_cache[analysis_id] = html_path
    return html_path, {**viewer_meta, 'viewer_cache_hit': False}


def _viewer_payload(preds: np.ndarray, input_type: str, metadata: dict, peak_step: int) -> dict:
    analysis_id = _analysis_id(preds, input_type, metadata)
    payload = {
        'analysis_id': analysis_id,
        'viewer_url': f'/viewer/{analysis_id}',
        'viewer_available': False,
    }
    try:
        _, viewer_meta = _build_viewer_html(preds, analysis_id, peak_step, input_type)
        payload.update(viewer_meta)
        payload['viewer_available'] = True
    except Exception as e:
        payload['viewer_error'] = str(e)
    return payload


def _predict(events):
    preds, segments = tribe_model.predict(events=events)
    return np.asarray(preds), _segments_json(segments)


def _activation_payload(
    preds: np.ndarray,
    input_type: str,
    segments: list | None = None,
    metadata: dict | None = None,
) -> dict:
    activation = preds.tolist()
    summary = _activation_summary(preds)
    metadata = metadata or {}
    viewer = _viewer_payload(preds, input_type, metadata, summary['peak_activation_step'])

    payload = {
        'input_type': input_type,
        'shape': list(preds.shape),
        'activation': activation,
        'allPreds': _viewer_preds(preds),
        'segments': segments if segments is not None else [],
        'summary': summary,
        'metadata': metadata,
        # Convenience top-level fields for clients that don't want to unpack summary.
        'scores': summary['scores'],
        'region_masks': summary['region_masks'],
        'peak_activation_step': summary['peak_activation_step'],
        'analysis_id': viewer['analysis_id'],
        'viewer_url': viewer['viewer_url'],
        'viewer_available': viewer['viewer_available'],
    }
    for key in (
        'viewer_error',
        'viewer_kind',
        'viewer_frame_count',
        'viewer_source_frame_count',
        'viewer_stride',
        'viewer_peak_frame',
        'viewer_cache_hit',
    ):
        if key in viewer:
            payload[key] = viewer[key]
    return payload


def _infer_text(text: str):
    normalized = text.strip()
    key = hashlib.sha256(normalized.encode('utf-8')).hexdigest()
    if key in _text_cache:
        return _text_cache[key]

    tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.txt', mode='w', encoding='utf-8')
    try:
        tmp.write(normalized)
        tmp.flush()
        os.fsync(tmp.fileno())
        tmp.close()
        events = tribe_model.get_events_dataframe(text_path=tmp.name)
    finally:
        if os.path.exists(tmp.name):
            os.unlink(tmp.name)

    result = _predict(events)
    _text_cache[key] = result
    return result


async def _save_upload(file: UploadFile, allowed_ext: set[str]) -> str:
    ext = os.path.splitext(file.filename or '')[1].lower()
    if ext not in allowed_ext:
        raise HTTPException(400, f'Unsupported format {ext}. Allowed: {", ".join(sorted(allowed_ext))}')
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=ext)
    try:
        tmp.write(await file.read())
        tmp.flush()
        os.fsync(tmp.fileno())
        tmp.close()
        return tmp.name
    except Exception:
        if os.path.exists(tmp.name):
            os.unlink(tmp.name)
        raise


@app.get('/health')
async def health():
    return {
        'status': 'ok',
        'model': 'facebook/tribev2',
        'api_version': TRIBE_API_VERSION,
        'response_fields': [
            'activation',
            'allPreds',
            'segments',
            'shape',
            'input_type',
            'metadata',
            'summary',
            'scores',
            'region_masks',
            'peak_activation_step',
            'analysis_id',
            'viewer_url',
            'viewer_available',
            'viewer_kind',
            'viewer_frame_count',
            'viewer_stride',
        ],
        'text_cache_entries': len(_text_cache),
        'viewer_cache_entries': len(_viewer_cache),
        'viewer_backend': 'threejs_fsaverage5' if _VIEWER_IMPORT_ERROR is None else 'unavailable',
        'viewer_max_frames': VIEWER_MAX_FRAMES,
        'viewer_dir': VIEWER_DIR,
    }


class TextActivationRequest(BaseModel):
    text: str
    campaign_name: Optional[str] = None


@app.post('/activate/text')
async def activate_text(req: TextActivationRequest):
    if not req.text.strip():
        raise HTTPException(400, 'text cannot be empty')
    preds, segments = _infer_text(req.text)
    metadata = {'campaign_name': req.campaign_name or 'Unnamed Campaign'}
    return JSONResponse(_activation_payload(preds, 'text', segments, metadata))


@app.post('/activate/audio')
async def activate_audio(
    file: UploadFile = File(...),
    campaign_name: str = Form('Unnamed Campaign'),
):
    path = await _save_upload(file, AUDIO_EXT)
    try:
        events = tribe_model.get_events_dataframe(audio_path=path)
        preds, segments = _predict(events)
    finally:
        if os.path.exists(path):
            os.unlink(path)
    metadata = {'campaign_name': campaign_name, 'filename': file.filename}
    return JSONResponse(_activation_payload(preds, 'audio', segments, metadata))


@app.post('/activate/video')
async def activate_video(
    file: UploadFile = File(...),
    campaign_name: str = Form('Unnamed Campaign'),
):
    path = await _save_upload(file, VIDEO_EXT)
    try:
        events = tribe_model.get_events_dataframe(video_path=path)
        preds, segments = _predict(events)
    finally:
        if os.path.exists(path):
            os.unlink(path)
    metadata = {'campaign_name': campaign_name, 'filename': file.filename}
    return JSONResponse(_activation_payload(preds, 'video', segments, metadata))


@app.get('/viewer/{analysis_id}')
async def get_viewer(analysis_id: str):
    path = _viewer_cache.get(analysis_id)
    if not path:
        candidate = os.path.join(VIEWER_DIR, f'{analysis_id}.html')
        path = candidate if os.path.exists(candidate) else None
    if not path or not os.path.exists(path):
        raise HTTPException(404, f'Viewer not found for analysis_id={analysis_id}')
    return FileResponse(path, media_type='text/html')


print(f'API v{TRIBE_API_VERSION} — responses include activation, allPreds, summary, scores, peak_activation_step, analysis_id, viewer_url')
print('Viewer: one-piece Three.js fsaverage5 brain; text is static peak frame, audio/video animate timesteps')
print('Endpoints: GET /health | GET /viewer/{analysis_id} | POST /activate/text | /activate/audio | /activate/video')

API v2.3 — responses include activation, allPreds, segments, shape, scores, peak_activation_step
Endpoints: GET /health | POST /activate/text | /activate/audio | /activate/video


In [4]:
import subprocess
import time

import nest_asyncio
import requests
import uvicorn
from pyngrok import ngrok
from threading import Thread

nest_asyncio.apply()

# Free port 8000 so re-running this cell picks up the latest API code
subprocess.run('fuser -k 8000/tcp 2>/dev/null || true', shell=True, check=False)
time.sleep(0.5)

for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

tunnel = ngrok.connect(8000, domain=NGROK_DOMAIN)
public_url = tunnel.public_url

Thread(
    target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning'),
    daemon=True,
).start()

time.sleep(1.5)
health = requests.get('http://127.0.0.1:8000/health', timeout=10).json()
expected_version = TRIBE_API_VERSION
if health.get('api_version') != expected_version:
    raise RuntimeError(
        f'Stale API on port 8000 (got {health.get("api_version")!r}, want {expected_version!r}). '
        'Restart kernel, then Run All from the install cell.'
    )

print('=' * 62)
print('TRIBE activation API is live')
print(f'  Base URL : {public_url}')
print(f'  API docs : {public_url}/docs')
print(f'  Version  : {health["api_version"]}  fields: {health["response_fields"]}')
print('=' * 62)
for ep in ['/health', '/activate/text', '/activate/audio', '/activate/video', '/viewer/{analysis_id}']:
    print(f'  {public_url}{ep}')
print('\nViewer test pattern: ' + public_url + '/viewer/<analysis_id from activate response>')
print('Set in frontend: VITE_TRIBE_API=' + public_url)
print('Server running — keep this cell alive.')

TRIBE activation API is live
  Base URL : https://sibling-luminous-gothic.ngrok-free.dev
  API docs : https://sibling-luminous-gothic.ngrok-free.dev/docs
  Version  : 2.3  fields: ['activation', 'allPreds', 'segments', 'shape', 'input_type', 'metadata', 'summary', 'scores', 'region_masks', 'peak_activation_step']
  https://sibling-luminous-gothic.ngrok-free.dev/health
  https://sibling-luminous-gothic.ngrok-free.dev/activate/text
  https://sibling-luminous-gothic.ngrok-free.dev/activate/audio
  https://sibling-luminous-gothic.ngrok-free.dev/activate/video

Set in frontend: VITE_TRIBE_API=https://sibling-luminous-gothic.ngrok-free.dev
Server running — keep this cell alive.


In [5]:
# Smoke test — saves FULL JSON for BrainViewer and validates interactive HTML viewer
import json
import os

import numpy as np
import requests
from IPython.display import FileLink, IFrame, display

base = 'http://127.0.0.1:8000'
health = requests.get(f'{base}/health', timeout=10).json()
print('Health:', health)
if health.get('api_version') != TRIBE_API_VERSION:
    raise RuntimeError(
        f'API version mismatch: {health.get("api_version")!r} != {TRIBE_API_VERSION!r}. '
        'Re-run the API cell + server cell (or Restart kernel → Run All).'
    )

r = requests.post(
    f'{base}/activate/text',
    json={'text': 'A short sentence for TRIBE.', 'campaign_name': 'Smoke Test Campaign'},
    timeout=1800,
)
r.raise_for_status()
result = r.json()


def _viewer_preds_from_activation(activation):
    """Per-frame 0–1 from |activation| (if API has no allPreds yet)."""
    arr = np.asarray(activation, dtype=np.float64)
    out = []
    for frame in arr:
        v = np.abs(frame)
        mx = float(v.max())
        out.append((v / mx).tolist() if mx > 0 else v.tolist())
    return out


required = (
    'activation',
    'allPreds',
    'segments',
    'shape',
    'input_type',
    'metadata',
    'summary',
    'scores',
    'peak_activation_step',
    'analysis_id',
    'viewer_url',
    'viewer_available',
    'viewer_kind',
    'viewer_frame_count',
)
missing = [k for k in required if k not in result]
if missing:
    if 'activation' in result and 'allPreds' not in result:
        result['allPreds'] = _viewer_preds_from_activation(result['activation'])
        result.setdefault('segments', [])
        result.setdefault('metadata', {})
        print('Warning: server missing allPreds — computed client-side; re-run API + server cells.')
    else:
        raise KeyError(f'API response missing fields: {missing}. Re-run API + server cells.')

api_path = '/kaggle/working/tribe_api_response.json'
with open(api_path, 'w', encoding='utf-8') as f:
    json.dump(result, f)

viewer_path = '/kaggle/working/brainviewer_payload.json'
viewer_payload = {
    'allPreds': result['allPreds'],
    'segments': result.get('segments', []),
    'shape': result['shape'],
    'input_type': result['input_type'],
    'metadata': result.get('metadata', {}),
    'campaign_name': result.get('metadata', {}).get('campaign_name'),
    'summary': result.get('summary', {}),
    'scores': result.get('scores', {}),
    'region_masks': result.get('region_masks', {}),
    'peak_activation_step': result.get('peak_activation_step'),
    'analysis_id': result.get('analysis_id'),
    'viewer_url': result.get('viewer_url'),
    'viewer_available': result.get('viewer_available', False),
    'viewer_kind': result.get('viewer_kind'),
    'viewer_frame_count': result.get('viewer_frame_count'),
    'viewer_source_frame_count': result.get('viewer_source_frame_count'),
    'viewer_stride': result.get('viewer_stride'),
    'viewer_peak_frame': result.get('viewer_peak_frame'),
}
with open(viewer_path, 'w', encoding='utf-8') as f:
    json.dump(viewer_payload, f)

# Compact binary
npy_path = '/kaggle/working/brainviewer_payload.npz'
np.savez_compressed(
    npy_path,
    allPreds=np.array(result['allPreds'], dtype=np.float32),
    activation=np.array(result['activation'], dtype=np.float32),
)

# Pull interactive HTML viewer from API and save as artifact
viewer_html_path = '/kaggle/working/brainviewer_interactive.html'
if result.get('viewer_available') and result.get('viewer_url'):
    viewer_resp = requests.get(f"{base}{result['viewer_url']}", timeout=300)
    viewer_resp.raise_for_status()
    with open(viewer_html_path, 'w', encoding='utf-8') as f:
        f.write(viewer_resp.text)
else:
    print('Warning: viewer generation unavailable:', result.get('viewer_error'))

print('input_type:', result['input_type'])
print('metadata:', result.get('metadata'))
print('shape:', result['shape'], '  (frames, vertices)')
print('segments:', len(result.get('segments', [])))
print('peak_activation_step:', result.get('peak_activation_step'))
print('analysis_id:', result.get('analysis_id'))
print('viewer_url:', result.get('viewer_url'))
print('viewer_available:', result.get('viewer_available'))
print('viewer_kind:', result.get('viewer_kind'))
print('viewer_frame_count:', result.get('viewer_frame_count'))
print('viewer_source_frame_count:', result.get('viewer_source_frame_count'))
print('viewer_stride:', result.get('viewer_stride'))
print('scores:', result.get('scores'))
print()

artifact_paths = [api_path, viewer_path, npy_path]
if os.path.exists(viewer_html_path):
    artifact_paths.append(viewer_html_path)

for path in artifact_paths:
    print(f'{path}  ({os.path.getsize(path) / 1e6:.2f} MB)')

print()
print('Download from Kaggle Output, or use in frontend:')
display(FileLink(api_path))
display(FileLink(viewer_path))
display(FileLink(npy_path))
if os.path.exists(viewer_html_path):
    display(FileLink(viewer_html_path))
    print('Inline preview (one-piece static/animated WebGL):')
    display(IFrame(src=viewer_html_path, width='100%', height=720))

print('Smoke test passed — text should be static peak-frame; audio/video should animate if multiple frames are returned')

Health: {'status': 'ok', 'model': 'facebook/tribev2', 'api_version': '2.3', 'response_fields': ['activation', 'allPreds', 'segments', 'shape', 'input_type', 'metadata', 'summary', 'scores', 'region_masks', 'peak_activation_step'], 'text_cache_entries': 0}


INFO - Wrote TTS audio to /kaggle/working/tribe_cache/tribev2.demo_utils.TextToEvents.get_events,0/text=A-short-sentence-for-TRIBE.-019e54a3/audio.mp3
Extracting words from audio: 100%|██████████| 1/1 [02:00<00:00, 120.31s/it]
/usr/local/lib/python3.12/dist-packages/neuralset/events/utils.py:134: UserWarning: The events dataframe contains an `Index` column. This is dangerous, please add drop=True in calls to df.reset_index(). Dropping it automatically.
  warnings.warn(msg)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 65.6 MB/s  0:00:04
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Add context to words: 100%|██████████| 5/5 [00:00<00:00, 28688.81it/s]
[00:23:51 WARNING] Removing extractor video as there are no corresponding events
[00:23:51 INFO] Preparing extractor: text
Computing word embeddings:   0%|          | 0/2 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

...

input_type: text
metadata: {'campaign_name': 'Smoke Test Campaign'}
shape: [3, 20484]   (frames, vertices)
segments: 3
peak_activation_step: 0
scores: {'visual_cortex': 40.5, 'language_network': 41.0, 'attention': 42.4, 'emotional_response': 39.3, 'memory_encoding': 38.0, 'overall_impact': 40.2}

/kaggle/working/tribe_api_response.json  (2.61 MB)
/kaggle/working/brainviewer_payload.json  (1.28 MB)
/kaggle/working/brainviewer_payload.npz  (0.45 MB)

Download from Kaggle Output, or use in frontend:


/kaggle/working/tribe_api_response.json

/kaggle/working/brainviewer_payload.json

/kaggle/working/brainviewer_payload.npz

Smoke test passed — use brainviewer_payload.json in BrainViewer


In [ ]:
# API + brainviewer_payload.json fields:
#   activation           — raw TRIBE output (frames × ~20484)
#   allPreds             — per-frame |value| normalized to 0–1 (pass to BrainViewer)
#   segments             — timeline segments from predict()
#   shape                — [n_frames, n_vertices]
#   metadata             — optional campaign_name / filename labels
#   scores               — approximate region scores from REGION_MASKS
#   peak_activation_step — frame with strongest mean absolute activation
#   summary              — scores + region_masks + peak_activation_step
#   analysis_id          — stable id for this activation result
#   viewer_url           — GET /viewer/{analysis_id} interactive HTML endpoint
#   viewer_available     — whether HTML viewer was generated successfully
#   viewer_kind          — threejs_one_piece_static for text/single frame, threejs_one_piece_animated for multi-frame
#   viewer_frame_count   — number of frames embedded in the HTML viewer
#   viewer_stride        — approximate source-frame stride when long media is capped